### 1. Carga de datos

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

esqueleto = pd.read_parquet('/content/drive/MyDrive/TFM/favorita_panel_corregido.parquet')
print(esqueleto.shape)

Mounted at /content/drive
(4804366, 22)


### 2. Preparación de variables

In [2]:
if 'type' in esqueleto.columns:
    esqueleto = esqueleto.rename(columns={'type': 'store_type'})

for col in ['store_nbr', 'item_nbr', 'city', 'state', 'store_type', 'cluster', 'class']:
    esqueleto[col] = esqueleto[col].astype(str)

### 3. Partición de datos

In [3]:
fecha_max = esqueleto['date'].max()
test_inicio = fecha_max - pd.Timedelta(days=60)
val_inicio = test_inicio - pd.Timedelta(days=60)

train = esqueleto[esqueleto['date'] < val_inicio]
val = esqueleto[(esqueleto['date'] >= val_inicio) & (esqueleto['date'] < test_inicio)]

print('Train:', train.shape, '| Val:', val.shape)

Train: (4322665, 22) | Val: (238860, 22)


### 4. Baseline naive

In [4]:
esqueleto['pred_naive'] = esqueleto.groupby(['store_nbr', 'item_nbr'])['unit_sales'].shift(7)

### 5. Ponderación por volumen

In [5]:
volumen_por_serie = esqueleto.groupby(['store_nbr', 'item_nbr'])['unit_sales'].transform('mean')
esqueleto['peso_muestra'] = 1 / np.sqrt(volumen_por_serie + 0.1)
esqueleto['peso_muestra'] = esqueleto['peso_muestra'] / esqueleto['peso_muestra'].mean()

### 6. Instalación e importaciones de PyTorch Forecasting

In [6]:
!pip install pytorch-forecasting pytorch-lightning lightning -q

from pytorch_forecasting import TimeSeriesDataSet, DeepAR
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import NegativeBinomialDistributionLoss
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping
import torch

print(torch.cuda.is_available())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.3/425.3 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 16.7 MB/s eta 0:00:00
True


### 7. Reconstrucción de TimeSeriesDataSet corregido

In [7]:
max_encoder_length = 90
max_prediction_length = 30
training_cutoff = train['time_idx'].max()

training = TimeSeriesDataSet(
    esqueleto[esqueleto.time_idx <= training_cutoff],
    time_idx="time_idx",
    target="unit_sales",
    group_ids=["store_nbr", "item_nbr"],
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,
    static_categoricals=["city", "state", "store_type", "cluster", "class"],
    time_varying_known_reals=["time_idx", "year", "month", "day_of_week", "is_weekend",
                               "es_feriado", "onpromotion", "edad"],
    time_varying_unknown_reals=["unit_sales"],
    target_normalizer=GroupNormalizer(groups=["store_nbr", "item_nbr"], center=False),
    weight="peso_muestra",
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=False,
)

validation = TimeSeriesDataSet.from_dataset(
    training, esqueleto, min_prediction_idx=training_cutoff + 1, stop_randomization=True
)

/usr/local/lib/python3.13/dist-packages/pytorch_forecasting/data/timeseries/_timeseries.py:1861: UserWarning: Min encoder length and/or min_prediction_idx and/or min prediction length and/or lags are too large for 88 series/groups which therefore are not present in the dataset index. This means no predictions can be made for those series. First 10 removed groups: [{'__group_id__store_nbr': '1', '__group_id__item_nbr': '1428779'}, {'__group_id__store_nbr': '1', '__group_id__item_nbr': '2002136'}, {'__group_id__store_nbr': '1', '__group_id__item_nbr': '2027777'}, {'__group_id__store_nbr': '1', '__group_id__item_nbr': '2027827'}, {'__group_id__store_nbr': '1', '__group_id__item_nbr': '2053590'}, {'__group_id__store_nbr': '1', '__group_id__item_nbr': '2053610'}, {'__group_id__store_nbr': '1', '__group_id__item_nbr': '2053614'}, {'__group_id__store_nbr': '10', '__group_id__item_nbr': '2033805'}, {'__group_id__store_nbr': '10', '__group_id__item_nbr': '2053590'}, {'__group_id__store_nbr': '1

### 8. Dataloaders

In [8]:
batch_size = 128
train_dataloader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=0)
val_dataloader = validation.to_dataloader(train=False, batch_size=batch_size * 2, num_workers=0)

### 9. Escala de naive para el periodo de entrenamiento para MASE/RMSSE

In [9]:
train_naive = train.copy()
train_naive['pred_naive'] = train_naive.groupby(['store_nbr', 'item_nbr'])['unit_sales'].shift(7)
train_naive = train_naive.dropna(subset=['pred_naive'])
train_naive['err_abs'] = (train_naive['unit_sales'] - train_naive['pred_naive']).abs()
train_naive['err_sq'] = (train_naive['unit_sales'] - train_naive['pred_naive']) ** 2

escala_series = train_naive.groupby(['store_nbr', 'item_nbr']).agg(
    escala_mae=('err_abs', 'mean'),
    escala_rmse=('err_sq', lambda x: x.mean() ** 0.5)
).reset_index()

### 10. Conjunto de test

In [10]:
test_cutoff = val['time_idx'].max()
testing = TimeSeriesDataSet.from_dataset(training, esqueleto, min_prediction_idx=test_cutoff + 1, stop_randomization=True)
test_dataloader = testing.to_dataloader(train=False, batch_size=batch_size * 2, num_workers=0)

## 11. Evaluación de modelos
### 11.1. Configuración con todos los cuantiles

In [11]:
import gc

def evaluar_modelo_completo(modelo, test_dataloader, nombre_col):
    predictions_q = modelo.predict(
        test_dataloader, mode="quantiles", return_y=True, return_index=True,
        trainer_kwargs=dict(accelerator="auto", logger=False)
    )
    quantiles = modelo.loss.quantiles
    print('Cuantiles del modelo:', quantiles)  # verifica que 0.5 esté en la lista

    y_true = predictions_q.y[0].cpu().numpy()
    y_pred_todos = predictions_q.output.cpu().numpy()
    indice = predictions_q.index
    n_ventanas, horizonte = y_true.shape

    filas = {
        'store_nbr': np.repeat(indice['store_nbr'].values, horizonte),
        'item_nbr': np.repeat(indice['item_nbr'].values, horizonte),
        'time_idx': np.repeat(indice['time_idx'].values, horizonte) + np.tile(np.arange(horizonte), n_ventanas),
        'actual': y_true.flatten(),
    }
    for i, q in enumerate(quantiles):
        filas[f'{nombre_col}_q{q}'] = y_pred_todos[..., i].flatten()

    resultados = pd.DataFrame(filas).drop_duplicates(subset=['store_nbr', 'item_nbr', 'time_idx'], keep='first')

    del predictions_q, y_true, y_pred_todos, indice
    gc.collect()
    torch.cuda.empty_cache()
    return resultados, quantiles

### 11.2. Evaluación de v7

In [12]:
deepar_v7 = DeepAR.load_from_checkpoint('/content/drive/MyDrive/TFM/deepar_v7_corregido.ckpt')
comp_v7, quantiles = evaluar_modelo_completo(deepar_v7, test_dataloader, 'v7')
comp_v7.to_parquet('/content/drive/MyDrive/TFM/pred_v7.parquet', index=False)
del deepar_v7, comp_v7
gc.collect(); torch.cuda.empty_cache()
print('v7 evaluado y guardado')

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to th

Cuantiles del modelo: [0.02, 0.1, 0.25, 0.5, 0.75, 0.9, 0.98]
v7 evaluado y guardado


### 11.3. Evaluación de v8

In [13]:
deepar_v8 = DeepAR.load_from_checkpoint('/content/drive/MyDrive/TFM/deepar_v8_corregido.ckpt')
comp_v8, _ = evaluar_modelo_completo(deepar_v8, test_dataloader, 'v8')
comp_v8.to_parquet('/content/drive/MyDrive/TFM/pred_v8.parquet', index=False)
del deepar_v8, comp_v8
gc.collect(); torch.cuda.empty_cache()
print('v8 evaluado y guardado')

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to th

Cuantiles del modelo: [0.02, 0.1, 0.25, 0.5, 0.75, 0.9, 0.98]
v8 evaluado y guardado


### 11.4. Evaluación de v9

In [14]:
deepar_v9 = DeepAR.load_from_checkpoint('/content/drive/MyDrive/TFM/deepar_v9_corregido.ckpt')
comp_v9, _ = evaluar_modelo_completo(deepar_v9, test_dataloader, 'v9')
comp_v9.to_parquet('/content/drive/MyDrive/TFM/pred_v9.parquet', index=False)
del deepar_v9, comp_v9
gc.collect(); torch.cuda.empty_cache()
print('v9 evaluado y guardado')

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to th

Cuantiles del modelo: [0.02, 0.1, 0.25, 0.5, 0.75, 0.9, 0.98]
v9 evaluado y guardado


In [15]:
# Unificar
pred_v7 = pd.read_parquet('/content/drive/MyDrive/TFM/pred_v7.parquet')
pred_v8 = pd.read_parquet('/content/drive/MyDrive/TFM/pred_v8.parquet')
pred_v9 = pd.read_parquet('/content/drive/MyDrive/TFM/pred_v9.parquet')

comparacion = pred_v7.merge(pred_v8.drop(columns='actual'), on=['store_nbr', 'item_nbr', 'time_idx']) \
    .merge(pred_v9.drop(columns='actual'), on=['store_nbr', 'item_nbr', 'time_idx']) \
    .merge(esqueleto[['store_nbr', 'item_nbr', 'time_idx', 'pred_naive']], on=['store_nbr', 'item_nbr', 'time_idx'], how='left')

comparacion.to_parquet('/content/drive/MyDrive/TFM/comparacion_completa_v2.parquet', index=False)
comparacion.shape

(242708, 26)

### 11.5. Métricas

In [16]:
# Métricas completas: MAE-RMSE-WAPE, pinball loss, calibración, MASE/RMSSE
def pinball_loss(actual, pred, q):
    e = actual - pred
    return np.maximum(q * e, (q - 1) * e).mean()

def metricas_completas(comp, prefijo, quantiles, escala_series):
    med = comp[f'{prefijo}_q0.5']
    mae = (comp['actual'] - med).abs().mean()
    rmse = ((comp['actual'] - med) ** 2).mean() ** 0.5
    wape = (comp['actual'] - med).abs().sum() / comp['actual'].sum()

    pinballs = [pinball_loss(comp['actual'], comp[f'{prefijo}_q{q}'], q) for q in quantiles]
    pinball_prom = np.mean(pinballs)

    cobertura_80 = ((comp['actual'] >= comp[f'{prefijo}_q0.1']) & (comp['actual'] <= comp[f'{prefijo}_q0.9'])).mean()
    cobertura_96 = ((comp['actual'] >= comp[f'{prefijo}_q0.02']) & (comp['actual'] <= comp[f'{prefijo}_q0.98'])).mean()

    c = comp.merge(escala_series, on=['store_nbr', 'item_nbr'], how='left').copy()
    c['err_abs'] = (c['actual'] - med).abs()
    c['err_sq'] = (c['actual'] - med) ** 2
    por_serie = c.groupby(['store_nbr', 'item_nbr']).agg(
        mae_serie=('err_abs', 'mean'), rmse_serie=('err_sq', lambda x: x.mean() ** 0.5),
        escala_mae=('escala_mae', 'first'), escala_rmse=('escala_rmse', 'first'),
    )
    por_serie['mase'] = por_serie['mae_serie'] / por_serie['escala_mae']
    por_serie['rmsse'] = por_serie['rmse_serie'] / por_serie['escala_rmse']
    por_serie = por_serie.replace([np.inf, -np.inf], np.nan)
    n_excluidas = por_serie['mase'].isna().sum()

    return {
        'MAE': mae, 'RMSE': rmse, 'WAPE': wape, 'pinball_loss': pinball_prom,
        'cobertura_80% (nominal 80%)': cobertura_80, 'cobertura_96% (nominal 96%)': cobertura_96,
        'MASE': por_serie['mase'].mean(), 'RMSSE': por_serie['rmsse'].mean(),
        'series_excluidas_escala_naive_cero': n_excluidas,
    }

resultados_v2 = pd.DataFrame([
    {'modelo': 'DeepAR v7', 'configuracion': 'hidden_size=30, batches=200', **metricas_completas(comparacion, 'v7', quantiles, escala_series)},
    {'modelo': 'DeepAR v8', 'configuracion': 'hidden_size=64, batches=200', **metricas_completas(comparacion, 'v8', quantiles, escala_series)},
    {'modelo': 'DeepAR v9', 'configuracion': 'hidden_size=30, batches=400', **metricas_completas(comparacion, 'v9', quantiles, escala_series)},
])
resultados_v2.to_csv('/content/drive/MyDrive/TFM/resultados_finales_v2.csv', index=False)
resultados_v2

,modelo,configuracion,MAE,RMSE,WAPE,pinball_loss,cobertura_80% (nominal 80%),cobertura_96% (nominal 96%),MASE,RMSSE,series_excluidas_escala_naive_cero
0,DeepAR v7,"hidden_size=30, batches=200",2.631415,5.311795,0.429460,0.730068,0.862176,0.958654,1.576975,0.709635,2
1,DeepAR v8,"hidden_size=64, batches=200",2.509670,5.145101,0.409591,0.705720,0.868327,0.956705,1.558711,0.703780,2
2,DeepAR v9,"hidden_size=30, batches=400",2.490041,5.055732,0.406387,0.694989,0.877594,0.963368,1.554106,0.698353,2


## 12. Test Diebold-Mariano

In [17]:
# Test con la mediana de cada modelo
from scipy import stats

def test_dm(comp, col_pred, tipo_perdida='L2'):
    d = comp.dropna(subset=['pred_naive', col_pred]).copy()
    if tipo_perdida == 'L2':
        d['err_naive'] = (d['actual'] - d['pred_naive']) ** 2
        d['err_modelo'] = (d['actual'] - d[col_pred]) ** 2
    else:
        d['err_naive'] = (d['actual'] - d['pred_naive']).abs()
        d['err_modelo'] = (d['actual'] - d[col_pred]).abs()
    d['diferencia'] = d['err_naive'] - d['err_modelo']
    dif_por_serie = d.groupby(['store_nbr', 'item_nbr'])['diferencia'].mean()
    n = len(dif_por_serie)
    dm_stat = dif_por_serie.mean() / (dif_por_serie.std(ddof=1) / np.sqrt(n))
    p_value = 2 * (1 - stats.norm.cdf(abs(dm_stat)))
    return n, dm_stat, p_value

for modelo, col in [('v7', 'v7_q0.5'), ('v8', 'v8_q0.5'), ('v9', 'v9_q0.5')]:
    for perdida in ['L2', 'L1']:
        n, dm_stat, p_value = test_dm(comparacion, col, perdida)
        print(f'DeepAR {modelo} vs. baseline ({perdida}) -> series: {n} | DM: {dm_stat:.4f} | p-valor: {p_value:.4g}')

DeepAR v7 vs. baseline (L2) -> series: 3981 | DM: 5.0336 | p-valor: 4.813e-07
DeepAR v7 vs. baseline (L1) -> series: 3981 | DM: 36.1852 | p-valor: 0
DeepAR v8 vs. baseline (L2) -> series: 3981 | DM: 5.9919 | p-valor: 2.074e-09
DeepAR v8 vs. baseline (L1) -> series: 3981 | DM: 46.2975 | p-valor: 0
DeepAR v9 vs. baseline (L2) -> series: 3981 | DM: 6.3273 | p-valor: 2.495e-10
DeepAR v9 vs. baseline (L1) -> series: 3981 | DM: 45.8992 | p-valor: 0
